# 01 — Scrape (smoke test)

Pulls a small sample of public GitHub repos that have a root-level `CLAUDE.md` or, failing that, an `AGENTS.md`, plus repo metadata, and checkpoints to Parquet.

**`N_TARGET` = 50 for the Kaiser pilot (20 was the smoke test).**

Scraping decisions (copy into the dataset card):
- **Unit = repo. `CLAUDE.md` takes priority; a repo with no `CLAUDE.md` falls back to its `AGENTS.md`** (`AGENTS.md` added with Kaiser's approval, 2026-11-04). `primary_file` records which group a repo is in. The sample is mostly `CLAUDE.md` repos; `FALLBACK_FRACTION` sets the `AGENTS.md`-only share (a design choice, not a measured population share).
- Query: `filename:<CLAUDE.md|AGENTS.md> path:/ size:<bucket>` via the code-search REST API, restricted to root-level files. Both searches feed one candidate pool; each repo's group is then decided from its actual root listing, not from which search found it.
- GitHub code search has no star/date filter, so candidates are gathered per **file-size bucket**, then repo metadata is fetched and the final sample is **stratified by group, then repo stars**.
- Caveat: size-bucketing deliberately over-samples unusual file lengths. Fine for a pipeline test; the full run needs a sampling scheme that does not distort length statistics.
- Excluded: forks, archived repos, files over 1 MB.
- `has_tests` = a test-like directory at the repo root (`test`, `tests`, `__tests__`, `spec`, `specs`). `has_ci` = `.github/workflows`, `.travis.yml`, `.circleci`, `.gitlab-ci.yml`, `azure-pipelines.yml` or `Jenkinsfile` at the root.
- Both files are fetched when a repo has both (`claude_text`, `agents_text`), plus `*_symlink_target` for symlinks. `content_raw` is the primary file's text. Redirect detection (`@AGENTS.md`, "see AGENTS.md") happens in `02_clean`.

In [1]:
import os, time, base64, datetime as dt
from pathlib import Path
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.environ.get("GITHUB_TOKEN")
assert TOKEN, "GITHUB_TOKEN missing from .env"

N_TARGET = 50                      # total repos in the sample
FALLBACK_FRACTION = 0.25           # share of the sample that is AGENTS.md-only repos (no CLAUDE.md)
FILE_TYPES = ["CLAUDE.md", "AGENTS.md"]
SEARCH_PAGE = 2                    # page 2 of code search = fresh repos vs. the first pilot
SEED = 7
PER_PAGE = 20                      # candidates fetched per (file type, size bucket)
SIZE_BUCKETS = ["<500", "500..2000", "2001..5000", "5001..20000", ">20000"]
STAR_BINS = [(0, 10), (10, 100), (100, 1000), (1000, 10**9)]
OUT_PATH = Path("data/raw") / f"pilot_{N_TARGET}.parquet"
SCRAPE_DATE = dt.date.today().isoformat()

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
})
print("scrape date:", SCRAPE_DATE)

scrape date: 2026-11-09


In [2]:
def gh(url, params=None, retries=5):
    """GET with rate-limit handling. Returns the Response (404 allowed), raises otherwise."""
    for attempt in range(retries):
        r = session.get(url, params=params, timeout=30)
        if r.status_code in (403, 429) and (
            r.headers.get("x-ratelimit-remaining") == "0" or "retry-after" in r.headers
            or "secondary rate" in r.text.lower()
        ):
            if "retry-after" in r.headers:
                wait = int(r.headers["retry-after"]) + 1
            else:
                reset = int(r.headers.get("x-ratelimit-reset", time.time() + 60))
                wait = max(reset - time.time(), 0) + 2
            wait = min(wait, 120)
            print(f"rate limited, sleeping {wait:.0f}s")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(2 ** attempt)
            continue
        if r.status_code == 404:
            return r
        r.raise_for_status()
        return r
    raise RuntimeError(f"gave up on {url}")

## 1. Search: one query per (file type, size bucket)

In [3]:
candidates, bucket_totals = [], {}
for ft in FILE_TYPES:
    for b in SIZE_BUCKETS:
        r = gh("https://api.github.com/search/code",
               {"q": f"filename:{ft} path:/ size:{b}", "per_page": PER_PAGE, "page": SEARCH_PAGE})
        d = r.json()
        bucket_totals[(ft, b)] = d["total_count"]
        for it in d["items"]:
            if it["path"] == ft:                           # root-level only, exact name
                candidates.append({"full_name": it["repository"]["full_name"],
                                   "file_type": ft, "size_bucket": b})
        time.sleep(7)                                      # code search: 10 req/min

cand = pd.DataFrame(candidates).drop_duplicates(["full_name", "file_type"])
print("total matches per (file type, size bucket) (GitHub's count, a snapshot):")
print(pd.Series(bucket_totals))
print(f"\n{len(cand)} candidate (repo, file type) rows, {cand.full_name.nunique()} unique repos")

total matches per (file type, size bucket) (GitHub's count, a snapshot):
CLAUDE.md  <500            36672
           500..2000       87680
           2001..5000     194560
           5001..20000    254976
           >20000          58496
AGENTS.md  <500            32832
           500..2000      130560
           2001..5000     213248
           5001..20000    238592
           >20000          40896
dtype: int64

197 candidate (repo, file type) rows, 197 unique repos


## 2. Repo metadata, exclusions, and each repo's group (from its root listing)

In [4]:
rows = []
for name in cand["full_name"].unique():
    r = gh(f"https://api.github.com/repos/{name}")
    if r.status_code == 404:
        continue
    m = r.json()
    rows.append({
        "full_name": name, "stars": m["stargazers_count"], "forks": m["forks_count"],
        "language": m["language"], "created_at": m["created_at"], "last_push_at": m["pushed_at"],
        "repo_size_kb": m["size"], "is_fork": m["fork"], "archived": m["archived"],
        "license": (m.get("license") or {}).get("spdx_id"),
    })
meta = pd.DataFrame(rows)
n0 = len(meta)
meta = meta[~meta.is_fork & ~meta.archived].reset_index(drop=True)
print(f"{n0} repos -> {len(meta)} after dropping forks/archived")

roots = {}                                                   # repo -> {root entry name: type}
for name in meta["full_name"]:
    root = gh(f"https://api.github.com/repos/{name}/contents/")
    roots[name] = {e["name"]: e["type"] for e in root.json()} if root.status_code == 200 else {}
meta["has_claude_md"] = [("CLAUDE.md" in roots[n]) for n in meta["full_name"]]
meta["has_agents_md"] = [("AGENTS.md" in roots[n]) for n in meta["full_name"]]
meta["has_both"] = meta["has_claude_md"] & meta["has_agents_md"]
meta["primary_file"] = [("CLAUDE.md" if c else "AGENTS.md" if a else None)
                        for c, a in zip(meta["has_claude_md"], meta["has_agents_md"])]
meta = meta[meta["primary_file"].notna()].reset_index(drop=True)   # drops search false-positives
meta = meta.merge(cand.drop_duplicates("full_name")[["full_name", "size_bucket"]], on="full_name")
print(meta["primary_file"].value_counts())

197 repos -> 197 after dropping forks/archived


primary_file
CLAUDE.md    132
AGENTS.md     65
Name: count, dtype: int64


## 3. Stratified sample: mostly CLAUDE.md repos, AGENTS.md-only as fallback, each stratified by stars

In [5]:
def star_bin(s):
    for lo, hi in STAR_BINS:
        if lo <= s < hi:
            return f"{lo}-{hi if hi < 10**9 else 'inf'}"
meta["star_bin"] = meta["stars"].map(star_bin)

n_fallback = round(N_TARGET * FALLBACK_FRACTION)
targets = {"CLAUDE.md": N_TARGET - n_fallback, "AGENTS.md": n_fallback}
parts = []
for pf, g in meta.groupby("primary_file"):
    want = targets[pf]
    per_bin = -(-want // len(STAR_BINS))                     # ceil
    pick = pd.concat([grp.sample(min(len(grp), per_bin), random_state=SEED)
                      for _, grp in g.groupby("star_bin")])
    if len(pick) < want:                                     # fill from the remainder
        rest = g.drop(pick.index)
        pick = pd.concat([pick, rest.sample(min(len(rest), want - len(pick)), random_state=SEED)])
    parts.append(pick.head(want))
sample = pd.concat(parts).reset_index(drop=True)
print(pd.crosstab(sample["primary_file"], sample["star_bin"]))

star_bin      0-10  10-100  100-1000
primary_file                        
AGENTS.md        2       5         5
CLAUDE.md        4      21        13


## 4. Fetch file content (both files when present) and repo-root signals

In [6]:
TEST_DIRS = {"test", "tests", "__tests__", "spec", "specs"}
CI_ROOT = {".travis.yml", ".circleci", ".gitlab-ci.yml", "azure-pipelines.yml", "jenkinsfile"}

def fetch_file(name, path):
    """Return (text, symlink_target). text is None if missing, a symlink, or >1 MB."""
    f = gh(f"https://api.github.com/repos/{name}/contents/{path}")
    if f.status_code != 200:
        return None, None
    j = f.json()
    if not isinstance(j, dict):
        return None, None
    if j.get("type") == "symlink":
        return None, j.get("target")
    if j.get("encoding") == "base64":
        return base64.b64decode(j["content"]).decode("utf-8", errors="replace"), None
    return None, None

claude_text, claude_link, agents_text, agents_link, has_tests, has_ci = [], [], [], [], [], []
for name in sample["full_name"]:
    entries = roots[name]
    lower = {n.lower(): t for n, t in entries.items()}
    ct, cl = fetch_file(name, "CLAUDE.md") if "CLAUDE.md" in entries else (None, None)
    at, al = fetch_file(name, "AGENTS.md") if "AGENTS.md" in entries else (None, None)
    claude_text.append(ct); claude_link.append(cl)
    agents_text.append(at); agents_link.append(al)

    has_tests.append(any(n in TEST_DIRS and t == "dir" for n, t in lower.items()))
    ci = any(n in CI_ROOT for n in lower)
    if ".github" in lower and not ci:
        wf = gh(f"https://api.github.com/repos/{name}/contents/.github/workflows")
        ci = wf.status_code == 200
    has_ci.append(ci)

sample["claude_text"], sample["claude_symlink_target"] = claude_text, claude_link
sample["agents_text"], sample["agents_symlink_target"] = agents_text, agents_link
is_claude = sample["primary_file"] == "CLAUDE.md"
sample["content_raw"] = sample["claude_text"].where(is_claude, sample["agents_text"])   # the primary file's text
sample["has_tests"] = has_tests
sample["has_ci"] = has_ci
sample["n_chars"] = sample["content_raw"].str.len()
sample["scraped_at"] = SCRAPE_DATE
primary_link = sample["claude_symlink_target"].where(is_claude, sample["agents_symlink_target"])
keep = sample["content_raw"].notna() | primary_link.notna()   # symlinks have no text of their own
sample = sample[keep].reset_index(drop=True)
print(len(sample), "repos kept;", sample["has_both"].sum(), "have both files;",
      (sample["claude_symlink_target"].notna() | sample["agents_symlink_target"].notna()).sum(), "have a symlink")

50 repos kept; 13 have both files; 0 have a symlink


## 5. Checkpoint to Parquet

In [7]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sample.to_parquet(OUT_PATH, index=False)
check = pd.read_parquet(OUT_PATH)
md_dir = OUT_PATH.with_name(OUT_PATH.stem + "_md")          # readable copies of the raw files
md_dir.mkdir(exist_ok=True)
for _, row in check.iterrows():
    if row["content_raw"] is not None:
        (md_dir / f"{row.full_name.replace('/', '__')}__{row.primary_file}").write_text(row["content_raw"])
print("saved", OUT_PATH, check.shape)
check.drop(columns=["content_raw", "claude_text", "agents_text"]).describe(include="all").T[["count", "unique", "top", "mean", "min", "max"]]

saved data/raw/pilot_50.parquet (50, 25)


,count,unique,top,mean,min,max
full_name,50,50,getsentry/image-mirror,NaN,NaN,NaN
stars,50.0,NaN,NaN,93.6,2.0,473.0
forks,50.0,NaN,NaN,14.94,0.0,98.0
language,49,17,TypeScript,NaN,NaN,NaN
created_at,50,50,2022-08-23T21:39:12Z,NaN,NaN,NaN
last_push_at,50,50,2026-11-05T09:05:54Z,NaN,NaN,NaN
repo_size_kb,50.0,NaN,NaN,42319.9,43.0,796927.0
is_fork,50,1,False,NaN,NaN,NaN
archived,50,1,False,NaN,NaN,NaN
license,48,11,MIT,NaN,NaN,NaN


In [8]:
check[["full_name", "primary_file", "stars", "language", "n_chars", "has_both",
       "claude_symlink_target", "has_tests", "has_ci", "star_bin"]]

,full_name,primary_file,stars,language,n_chars,has_both,claude_symlink_target,has_tests,has_ci,star_bin
0,getsentry/image-mirror,AGENTS.md,9,Python,801,False,None,False,True,0-10
1,sous-chefs/xinetd,AGENTS.md,5,Ruby,3760,False,None,True,True,0-10
2,zcaudate-xyz/foundation-base,AGENTS.md,42,Clojure,22033,False,None,True,True,10-100
3,NachoSoto/AsyncImageView,AGENTS.md,82,Swift,202,False,None,False,True,10-100
4,linuxserver/docker-budge,AGENTS.md,62,Dockerfile,14154,False,None,False,True,10-100
5,huang-sh/PiX,AGENTS.md,237,TypeScript,353,False,None,True,True,100-1000
6,aashaexo/soundshuman,AGENTS.md,306,JavaScript,2228,False,None,True,True,100-1000
7,gripebomb/ThreatDeck,AGENTS.md,111,Rust,425,False,None,False,True,100-1000
8,tin-computer/with-md,AGENTS.md,76,TypeScript,3941,False,None,False,False,10-100
9,vmDeshpande/ai-agent-automation,AGENTS.md,178,TypeScript,13903,False,None,False,True,100-1000
